# Environment

In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import re
import matplotlib.pyplot as plt

import data_preprocessing.kbase as kb 

from pathlib import Path
from typing import List, Tuple, Dict

import matplotlib.pyplot as plt

import os

El directorio de trabajo actual es: /home/alvaro.ritore.ext/projects/trIAje/analysis
El directorio de trabajo actual es: /media/datos/datos_compartidos/trIAje/data


# Functions

# Main Vectors

In [ ]:
key_col = "demandpk"
id_columns = ["healthcard_id", "national_id", "full_name"]

# Data Loading

## EDATOSCASO-LLAMADAS

In [ ]:
# Loading exported table
df_triaje = pq.read_table(os.path.join(kb.source_tables_path, "tabla_madre/master_table_4demands.parquet")).to_pandas()
df_triaje.shape

In [ ]:
df_triaje.info()

In [ ]:
df_triaje.head(5)


# Run Pipeline in subet

In [ ]:
# Flag to test the pipeline in a subset of 100K
## must be either "all", to run in all dataframe, or "test", to run it in the first 100K rows
df_triaje = kb.df_pipeline_test(df_triaje, "all").sort_values(by="DEMANDAPK")

# Column Reduction


In [ ]:
# IDTIPODEMANDA1 and IDTIPODEMANDA2: additional information outside of the scope

# Columns to drop
columns_to_drop = ['IDTIPODEMANDA2', 'IDTIPODEMANDA3'] 

# Drop columns
df_triaje = df_triaje.drop(columns = columns_to_drop)
# df_triaje.info()

# Data cleaning

## Rename columns

In [ ]:
df_triaje = kb.rename_columns(df_triaje)


In [ ]:
kb.summarize_numbers(df_triaje, key_col, id_columns)


In [ ]:
df_triaje.head(1)

## FECHADEMANDA - demand_date

In [ ]:
# Convert the raw string date
df_triaje["demand_date"] = pd.to_datetime(
    df_triaje["demand_date"],
    format="%Y%m%d",
    errors="coerce"
)

# Count how many failed conversions 
print(f'Number of invalid "demand_date" rows: {df_triaje["demand_date"].isna().sum()}')

df_triaje["demand_date"].dtype



## FNACIMIENTO - birth_date

In [ ]:
# Convert the raw string date
df_triaje["birth_date"] = pd.to_datetime(
    df_triaje["birth_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Count how many failed conversions 
print(f'Number of invalid "birth_date" rows: {df_triaje["birth_date"].isna().sum()}')

df_triaje["birth_date"].dtype

## EDAD - age

In [ ]:
df_triaje["age_type"].value_counts(dropna = False)

In [ ]:
df_triaje["age"].value_counts(dropna=False).sort_index(ascending=True)

In [ ]:
mask = (
    df_triaje["birth_date"].isna() &             # birth_date es NaT
    df_triaje["age"].notna() &              # call_age NO es NaN
    (df_triaje["age"] != 0)                 # call_age NO es 0
)

mask.sum()



In [ ]:

def process_age_pipeline(df, birth_col="birth_date", demand_col="demand_date",
                         age_col="age", age_type_col="age_type"):
    """
    Full age processing pipeline applied directly on `age` column.

    Steps:
    1. Compute age from birth_date when available.
    2. Compute age from reported age + age_type (A/M/D/S) when birth_date is missing.
    3. Replace the original 'age' column with the best available estimate.
    4. Round age to nearest integer.
    5. Convert final age to Int8.
    6. Remove auxiliary processing columns.

    Returns:
        The dataframe with an updated 'age' column (Int8).
    """

    # ------------------------------------------------------------
    # Step 1 — Compute age in years using birth_date
    # ------------------------------------------------------------
    df["age_from_birthdate"] = (
        (df[demand_col] - df[birth_col])
        .dt.total_seconds() / (365.25 * 24 * 3600)
    )

    # ------------------------------------------------------------
    # Step 2 — Define conversion multipliers for age_type
    # ------------------------------------------------------------
    unit_to_years = {
        "A": 1.0,        # years
        "M": 1/12,       # months
        "D": 1/365.25,   # days
        "S": 1/52,       # weeks
    }

    # ------------------------------------------------------------
    # Step 3 — Normalize age_type into uppercase strings
    # ------------------------------------------------------------
    df[age_type_col] = df[age_type_col].astype("string").str.upper()

    # ------------------------------------------------------------
    # Step 4 — Compute reported age in years using age and age_type
    # ------------------------------------------------------------
    df["reported_age_years"] = df[age_col] * df[age_type_col].map(unit_to_years)

    # ------------------------------------------------------------
    # Step 5 — Mask for rows missing birth_date but having reported age
    # ------------------------------------------------------------
    mask_missing_birth = (
        df[birth_col].isna() &
        df[age_col].notna() &
        (df[age_col] != 0)
    )

    # ------------------------------------------------------------
    # Step 6 — Update age directly
    # ------------------------------------------------------------
    # Start with age_from_birthdate when available
    df[age_col] = df["age_from_birthdate"]

    # Replace missing birthdate ages with reported_age_years
    df.loc[mask_missing_birth, age_col] = df.loc[mask_missing_birth, "reported_age_years"]

    # ------------------------------------------------------------
    # Step 7 — Round final age to nearest integer
    # ------------------------------------------------------------
    df[age_col] = df[age_col].round()

    # ------------------------------------------------------------
    # Step 8 — Convert to Int16
    # ------------------------------------------------------------
    df[age_col] = df[age_col].astype("Int16")

    # ------------------------------------------------------------
    # Step 9 — Remove auxiliary columns
    # ------------------------------------------------------------
    df = df.drop(columns=["age_from_birthdate", "reported_age_years"])

    return df

df_triaje = process_age_pipeline(df_triaje)


In [ ]:
mask = (
    df_triaje["birth_date"].isna() &             # birth_date es NaT
    df_triaje["age"].notna() &              # call_age NO es NaN
    (df_triaje["age"] != 0)                 # call_age NO es 0
)

mask.sum()

In [ ]:
## Exclusion of outliers and filtering by age range of interest
nrows_before_age_exclusion = len(df_triaje)
print(f"Number of rows before age filtering: {nrows_before_age_exclusion}")

df_triaje = df_triaje[df_triaje["age"].between(0, 105)]

nrows_after_age_exclusion = len(df_triaje)
print(f"Number of rows after age filtering: {nrows_after_age_exclusion}")

print("Rows removed after excluding outliers and filtering by age range:",
      nrows_before_age_exclusion - nrows_after_age_exclusion)


In [ ]:
# df_triaje_bebe = df_triaje[df_triaje["age"]==0]

In [ ]:
# df_triaje_bebe.head(100)

In [ ]:
df_triaje["age"].value_counts(dropna=False).sort_index().reset_index()


In [ ]:
df_triaje['age'].hist(bins=20, range=(0, 100), color='skyblue', edgecolor='black')
plt.title('Histogram of Age')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.6) 
plt.show()

df_triaje.boxplot(column=['age'], by='demand_type_1')
plt.title('Boxplot of Age')
plt.ylabel('Age')
plt.show()


## SEX - sex

In [ ]:
print("Value counts of 'sex' column before conversion:")
print(df_triaje['sex'].value_counts(dropna=False))

In [ ]:
# Map dictionary
## 'D' and None values are unknown
## Male (H): 0
## Female (M): 1
map_sex = {
    'D': np.nan,
    'None': np.nan,
    'H': 0,
    'M': 1
}

# Map the values
df_triaje["sex"] = df_triaje["sex"].map(map_sex)

# Convert data type
df_triaje["sex"] = pd.to_numeric(df_triaje["sex"], errors="coerce")
df_triaje["sex"] = df_triaje["sex"].astype("Int64")
df_triaje["sex"] = df_triaje["sex"].astype("category")

# Check the mapping
print("\nValue counts of 'sex' column after data conversion:")
df_triaje["sex"].value_counts(dropna=False)


In [ ]:
## Exclusion of None
nrows_before_sex_exclusion = len(df_triaje)
print(f"Number of rows before filtering out missing values in sex: {nrows_before_sex_exclusion}")

# Filter out rows where sex is NaN
df_triaje = df_triaje[df_triaje["sex"].notna()]

nrows_after_sex_exclusion = len(df_triaje)
print(f"Number of rows after filtering out missing values in sex: {nrows_after_sex_exclusion}")

print(
    "Rows removed after filtering out missing values in sex:",
    nrows_before_sex_exclusion - nrows_after_sex_exclusion
)


In [ ]:
print("\nValue counts of 'sex' column after removing missing values:")
df_triaje["sex"].value_counts(dropna=False)


## Time columns

In [ ]:
# List of time columns
time_columns = [
    "contact_entry_time",
    "contact_answer_time",
    "activation_time",
    "departure_time",
    "arrival_time",
    "load_time",
    "operative_time",
    "destination_time",
    "available_time"
]

for col in time_columns:
    df_triaje[col] = pd.to_datetime(
        df_triaje[col],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )

# Count NaT values after conversion
print("\nCount of invalid or missing datetime values after conversion:")
for col in time_columns:
    nat_count = df_triaje[col].isna().sum()
    dtype = df_triaje[col].dtype
    print(f"{col} ({dtype}): {nat_count} NaT values")


## IDPROVINCIA - province

In [ ]:
# Original value counts
print("Value counts of 'province' column before conversion:")
df_triaje["province"].value_counts(dropna=False)

In [ ]:
# Map dictionary
map_province = {
    4 : "ALM",
    11: "CAD",
    14: "COR",
    18: "GRA",
    21: "HUE",
    23: "JAE",
    29: "MAL",
    41: "SEV"
}

# Map the values
df_triaje["province"] = df_triaje["province"].map(map_province)

# Check the mapping
print("Value counts of 'province' column after conversion:")
df_triaje["province"].value_counts(dropna=False)


## resource_dest_province_id

In [ ]:
# Original value counts
print("Value counts of 'resource_dest_province_id' column before conversion:")
df_triaje["resource_dest_province_id"].value_counts(dropna=False)


## IDALERTANTE - alert_receiver

This variable provides us with an identification code for the entity that received the alert. We have a total of 23 different IDs (as we will verify below). According to clinical indications, they will be regrouped into the following 6 groups:

- GROUP 1 '112': '112'.
- GROUP 2 'USER': 'USER' (User).
- GROUP 3 'POL/FG' (Police/Fire Brigade): '91', '92', '62', '80', '091', '092', '062', '080'.
- GROUP 4 'HS' (Health System): 'APS', 'SEU', 'SNU', 'HOS', '61', '061', 'ECA', 'PAC', 'DCC'.
- GROUP 5 'TELE' (Teleassistance): 'SAT'.
- GROUP 6 'OTHERS': 'PCI', 'OTR', 'SR', 'CRE', 'FAX', 'TEL', 'INA', 'RES'.

In [ ]:
# Original value counts
print("Value counts of 'alert_receiver' column before conversion:")
df_triaje["alert_receiver"].value_counts(dropna=False)

In [ ]:
# Map raw alert receiver codes to grouped categories.
def map_alert_receiver(x):
    """
    This function:
    - Normalizes input values (strip spaces, uppercase).
    - Handles None/NaN safely.
    - Uses a dictionary-based mapping for maintainability.
    - Returns np.nan when the code is unknown or invalid.
    """

    # If null → return NaN
    if pd.isna(x):
        return np.nan

    # Normalize input (string only)
    x = str(x).strip().upper()
    if x == "":
        return np.nan

    # Dictionary of mappings
    mapping_dict = {
        # Group 1
        "112": "112",

        # Group 2
        "USU": "USER",

        # Group 3: POL/FG
        "91": "POL/FG", "92": "POL/FG", "62": "POL/FG", "80": "POL/FG",
        "091": "POL/FG", "092": "POL/FG", "062": "POL/FG", "080": "POL/FG",

        # Group 4: HS
        "APS": "HS", "SEU": "HS", "SNU": "HS", "HOS": "HS",
        "61": "HS", "061": "HS", "ECA": "HS", "PAC": "HS", "DCC": "HS",

        # Group 5: TELE
        "SAT": "TELE",

        # Group 6: OTHERS
        "PCI": "OTHERS", "OTR": "OTHERS", "SR": "OTHERS", "CRE": "OTHERS",
        "FAX": "OTHERS", "TEL": "OTHERS", "INA": "OTHERS", "RES": "OTHERS",
    }

    # Return mapped value or NaN if not found
    return mapping_dict.get(x, np.nan)


# Map the values
df_triaje["alert_receiver"] = df_triaje["alert_receiver"].apply(map_alert_receiver)

# Check the mapping
print("Value counts of 'alert_receiver' column after conversion:")
df_triaje["alert_receiver"].value_counts(dropna=False)

## IDESCENARIO - location_patient
- GROUP 1 'HO' (Home): 'DO'
- GROUP 2 'PR' (Public Road): 'VP'
- GROUP 3 'OTHERS': 'PI', 'CA', 'OT'.

In [ ]:
# Original value counts
print("Value counts of 'location_patient' column before conversion:")
df_triaje["location_patient"].value_counts(dropna=False)

In [ ]:
# Map dictionary
map_location = {
    "DO": "HO",
    "VP": "PR",
    "CA": "OTHERS",
    "OT": "OTHERS",
    "PI": "OTHERS"
}

# Map the values
df_triaje["location_patient"] = df_triaje["location_patient"].map(map_location)

# Check the mapping
print("Value counts of 'location_patient' column after conversion:")
df_triaje["location_patient"].value_counts(dropna=False)

## IDTIPORECURSO - resource_type

In [ ]:
# Original value counts
print("Value counts of 'resource_type' column before conversion:")
df_triaje["resource_type"].value_counts(dropna=False)



In [ ]:
print(f"\nNumber of unique values in resource_type: {df_triaje['resource_type'].nunique()}")

## IDTARJETACORAZON - healthcard

In [ ]:
df_triaje["healthcard"].value_counts(dropna = False)

In [ ]:
print(f"Number of None values in 'healthcard' is: {df_triaje['healthcard'].isna().sum()}")
df_triaje["healthcard"] = df_triaje["healthcard"].notna().astype(int)
df_triaje["healthcard"].dtype

In [ ]:
df_triaje["healthcard"].value_counts(dropna = False)

## IDTARJETAAIRE - aircard

In [ ]:
df_triaje["aircard"].value_counts(dropna = False)

In [ ]:
print(f"Number of None values in 'aircard' is: {df_triaje['aircard'].isna().sum()}")
df_triaje["aircard"] = df_triaje["aircard"].notna().astype(int)
df_triaje["aircard"].dtype

In [ ]:
df_triaje["aircard"].value_counts(dropna = False)

## ESVULNERABLE - is_vulnerable

In [ ]:
df_triaje["is_vulnerable"].value_counts(dropna = False)

In [ ]:
df_triaje["is_vulnerable"] = df_triaje["is_vulnerable"].notna().astype(int)
df_triaje["is_vulnerable"].dtype

In [ ]:
df_triaje["is_vulnerable"].value_counts(dropna = False)

## IDHCDM

In [ ]:
df_triaje["hcdm_id"].isna().sum()

In [ ]:
df_triaje["hcdm_id"].notna().sum()

## Tests

In [ ]:
df_triaje.info()

## Transform data types

In [ ]:
df_triaje = kb.transform_column_dtypes(df_triaje)

In [ ]:
df_triaje.info()

In [ ]:
df_triaje.head(5)

# Creation of new variables

## Juicio clínico - icd

In [ ]:
cols_icd = ["icd1", "icd2", "icd3"]

df_triaje[cols_icd] = (
    df_triaje[cols_icd]
    .apply(lambda col: col.astype("string").str.strip())     # strip whitespace
    .replace(r'^\s*$', pd.NA, regex=True)                    # empty strings → NA
    .replace(["nan", "NaN", "None", "none"], pd.NA)          # textual nulls → NA
    .astype("string")                                        # final enforcement
)


In [ ]:
df_triaje.head(10)

In [ ]:
def combine_clean_columns(df, columns, sep):
    """
    Clean ICD-related columns and combine valid entries into a single string.
    Missing values are returned as pd.NA.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.
    columns : list
        List of ICD columns to clean and combine.
    sep : str
        Separator used to join non-missing cleaned values.
    """

    def clean_value(x):
        """Normalize different missing representations to None."""
        if pd.isna(x):  # Detects real NaN/None/NaT
            return None
        if isinstance(x, str):
            x_strip = x.strip()
            # Remove empty strings and stringified missing markers
            if x_strip.lower() in ["", "nan", "none"]:
                return None
            return x_strip
        return x

    # Apply cleaning column-wise (avoid deprecated applymap)
    cleaned = df[columns].apply(lambda col: col.apply(clean_value))

    # Combine valid entries using the provided separator
    combined = cleaned.apply(
        lambda row: sep.join(str(x) for x in row if x is not None),
        axis=1
    )

    # Convert empty strings to real missing values
    combined = combined.replace("", pd.NA)

    return combined

df_triaje["icd_emerg_combined"] = combine_clean_columns(
    df_triaje,
    columns=["icd1", "icd2", "icd3"],
    sep=" | "       # You choose the separator manually
)

# Transform dtype and reorder dataframe
df_triaje = kb.transform_column_dtypes(df_triaje)


In [ ]:
df_triaje.head(10)

In [ ]:
df_triaje_icd = df_triaje[df_triaje["icd_emerg_combined"].str.contains("47.9", na=False)]



In [ ]:
df_triaje_icd.head(10)

In [ ]:
df_triaje['icd_emerg_combined'].isna().sum()

In [ ]:
kb.summarize_numbers(df_triaje, key_col, id_columns)

In [ ]:
df_triaje.loc[
    df_triaje["icd_emerg_combined"].str.contains("798", na=False),
    "resolution_code_id"
].value_counts(dropna=False)


In [ ]:
df_triaje.info()

In [ ]:
df_triaje.head(10)

## P1 Predicted

In [ ]:
print("Value counts of 'p1_predicted' column before conversion:")
df_triaje['p1_predicted'].value_counts(dropna = False)

In [ ]:
print(df_triaje['p1_predicted'].dtype)

In [ ]:
# p1_predicted is the priority predicted during the call, assigned by the system
# 1 is maximum priority, 0 is non-maximum priority
df_triaje['p1_predicted'] = np.where(df_triaje['p1_predicted'] == '1', '1', '0')

In [ ]:
print("Value counts of 'p1_predicted' column after conversion:")
df_triaje['p1_predicted'].value_counts(dropna = False)

In [ ]:
print(df_triaje['p1_predicted'].dtype)

## P1 Real

In [ ]:
icd_table = pd.read_csv(
    os.path.join(kb.source_tables_path, "icd/icd_p1outcomes.tsv"),
    dtype=str,
    encoding="utf-8",
    sep="\t"
)


In [ ]:
icd_table.head(10)

In [ ]:
icd_table["AND/Exclusion"].value_counts()


In [ ]:
icd_table.info()

In [ ]:
print(len(icd_table))
icd_table["ICD_original"].nunique()

In [ ]:
def process_icd_codes(df, icd_combined_col, reference_table, reference_icd_col):
    """
    Process combined ICD codes to detect ICD exclusions and P1 priority flags.

    Parameters
    ----------
    df : pandas.DataFrame
        Main dataset containing the combined ICD codes column.
    icd_combined_col : str
        Name of the column containing combined ICD codes separated by "|".
    reference_table : pandas.DataFrame
        ICD reference table containing ICD codes, P1 labels, and Exclusion labels.
    reference_icd_col : str
        Name of the ICD column inside the reference table (e.g., 'ICD_original').

    Returns
    -------
    df_filtered : pandas.DataFrame
        Filtered dataframe with a new column 'p1_real'.
    stats : dict
        Dictionary with:
            - 'n_rows_final': number of rows after filtering exclusions
            - 'n_excluded_icd': number of rows excluded due to ICD exclusion rule
    """

    # ===============================================================
    # 1. Normalize ICD reference table and create lookup dictionaries
    # ===============================================================
    ref = reference_table.copy()

    # Standardize ICD column
    ref[reference_icd_col] = ref[reference_icd_col].astype("string").str.strip()

    # Create P1 flag (only explicit "P1")
    ref["p1_real"] = ref["P1 outcomes"].apply(
        lambda x: 1 if isinstance(x, str) and x.strip().upper() == "P1" else 0
    )

    # Create exclusion flag (only explicit "Exclusion")
    ref["exclusion_flag"] = ref["AND/Exclusion"].apply(
        lambda x: 1 if isinstance(x, str) and x.strip().lower() == "exclusion" else 0
    )

    # Lookup dictionaries for fast matching
    p1_map = dict(zip(ref[reference_icd_col], ref["p1_real"]))
    exclusion_map = dict(zip(ref[reference_icd_col], ref["exclusion_flag"]))


    # ===============================================================
    # 2. Function to evaluate a single row of combined ICD codes
    # ===============================================================
    def evaluate_icd_row(text):
        """Evaluate exclusion and P1 status for a given combined ICD string."""

        if pd.isna(text):
            return {"exclude": False, "p1": 0}

        # Split combined ICD string into individual codes
        raw_codes = [c.strip() for c in str(text).split("|")]
        codes = [c for c in raw_codes if c != ""]

        # Rule 1 — If any ICD is marked as Exclusion → exclude row
        for code in codes:
            if code in exclusion_map and exclusion_map[code] == 1:
                return {"exclude": True, "p1": None}

        # Rule 2 — If any ICD has P1 flag → mark P1
        for code in codes:
            if code in p1_map and p1_map[code] == 1:
                return {"exclude": False, "p1": 1}

        # Rule 3 — No P1 found → mark 0
        return {"exclude": False, "p1": 0}


    # ===============================================================
    # 3. Apply evaluation row-wise across the dataset
    # ===============================================================
    evaluation = df[icd_combined_col].apply(evaluate_icd_row)

    df["p1_real"] = evaluation.apply(lambda r: r["p1"])
    df["exclude_row"] = evaluation.apply(lambda r: r["exclude"])

    # Count how many rows will be excluded
    n_excluded_icd = df["exclude_row"].sum()

    # ===============================================================
    # 4. Remove excluded rows
    # ===============================================================
    df_filtered = df[df["exclude_row"] == False].drop(columns=["exclude_row"])

    n_rows_final = len(df_filtered)

    
    # ===============================================================
    # 5. Convert p1_real to integer (0/1)
    # ===============================================================
    df_filtered["p1_real"] = df_filtered["p1_real"].astype("int64")
    # ---------------------------------------------------------------

    # ===============================================================
    # 6. Return both the filtered df and exclusion stats
    # ===============================================================
    stats = {
        "n_rows_final": n_rows_final,
        "n_excluded_icd": n_excluded_icd,
    }

    return df_filtered, stats

# Call the function
df_triaje, icd_stats = process_icd_codes(
    df=df_triaje,
    icd_combined_col="icd_emerg_combined",
    reference_table=icd_table,
    reference_icd_col="ICD_original"
)

# Extract values as independent variables
n_rows_final = icd_stats["n_rows_final"]
n_excluded_icd = icd_stats["n_excluded_icd"]

print(f"Total rows after ICD exclusion: {n_rows_final}")
print(f"Rows excluded due to ICD rules: {n_excluded_icd}")


In [ ]:
df_triaje["p1_real"].value_counts(dropna=False)

In [ ]:
df_triaje[df_triaje["icd_emerg_combined"].notna()].head(100)

## Date/Time variables

In [ ]:
# Create the day of the week (number, Monday=0, Sunday=6) variable
df_triaje['day_week'] = df_triaje['demand_date'].dt.dayofweek

In [ ]:
df_triaje['day_week'].value_counts(dropna = False)

In [ ]:
# Create the month (number) variable
df_triaje['month'] = df_triaje['demand_date'].dt.month

In [ ]:
df_triaje['month'].value_counts(dropna = False)

In [ ]:
# Create the year variable
df_triaje['year'] = df_triaje['demand_date'].dt.year

In [ ]:
df_triaje['year'].value_counts(dropna = False)

In [ ]:
# Display the new columns and the original date column
print(df_triaje[['demand_date', 'day_week', 'month', 'year']].head())

In [ ]:
def add_season_variable(df, date_col, output_col="season"):
    """
    Add a 'season' categorical column to the dataframe based on a date column.
    Seasons are defined for the Northern Hemisphere using fixed cutoff dates:
        - Spring: 21/03–20/06
        - Summer: 21/06–22/09
        - Autumn: 23/09–20/12
        - Winter: rest of the dates
    
    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    date_col : str
        Name of the datetime column used to compute the season.
    output_col : str, default="season"
        Name of the resulting season column.
    """

    # Compute MMDD as a single integer (e.g., March 21 -> 321)
    md = (df[date_col].dt.month * 100 + df[date_col].dt.day).astype("Int64")

    # Start assigning everything to "winter"
    season = pd.Series("winter", index=df.index, dtype="object")

    # Replace by season based on the date ranges
    season = season.mask(md.between(321, 620, inclusive="both"), "spring")
    season = season.mask(md.between(621, 922, inclusive="both"), "summer")
    season = season.mask(md.between(923, 1220, inclusive="both"), "autumn")

    # Preserve NA when the date itself is NA
    season = season.where(md.notna(), np.nan)

    # Ordered category
    season_order = ["spring", "summer", "autumn", "winter"]
    df[output_col] = pd.Categorical(season, categories=season_order, ordered=True)

    return df

df_triaje = add_season_variable(df_triaje, "demand_date", output_col="season")


In [ ]:
df_triaje["season"].value_counts()

In [ ]:
def add_time_slot(df, datetime_col, output_col="time_of_day"):
    """
    Create a categorical time-of-day slot from a datetime column.

    Time slots (0–23 hours):
        - early_morning : 00–05
        - morning       : 06–11
        - afternoon     : 12–17
        - night         : 18–23

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    datetime_col : str
        Name of the datetime column.
    output_col : str, default='timeslot'
        Name of the resulting categorical time-slot column.

    Returns
    -------
    df : pandas.DataFrame
        Dataframe with the new ordered categorical time-slot variable.
    """

    # Extract hour safely
    hour = df[datetime_col].dt.hour

    # Inner function for classification
    def _classify(h):
        if pd.isna(h):
            return np.nan
        if 0 <= h <= 5:
            return "early_morning"
        elif 6 <= h <= 11:
            return "morning"
        elif 12 <= h <= 17:
            return "afternoon"
        else:
            return "night"

    # Apply classification
    df[output_col] = hour.map(_classify)

    # Define ordered categories
    order = ["early_morning", "morning", "afternoon", "night"]
    df[output_col] = pd.Categorical(df[output_col], categories=order, ordered=True)

    return df


In [ ]:
df_triaje = add_time_slot(df_triaje, datetime_col="contact_entry_time")


In [ ]:
df_triaje["time_of_day"].value_counts()

## Transform data types

In [ ]:
df_triaje = kb.transform_column_dtypes(df_triaje)

In [ ]:
df_triaje.info()

In [ ]:
df_triaje.head(10)

# EDA

## Tests

In [ ]:
icd_cols = ["icd1", "icd2", "icd3"]

df_filtered = kb.filter_rows_by_icd(
    df=df_triaje,
    icd_value="047.9",
    icd_columns=icd_cols
)

df_filtered.head()

## demandpk–patientid Frequency Analysis

In [ ]:
# Step 1: Count how many unique patients are linked to each demandpk
demand_patient_counts = (
    df_triaje.groupby('demandpk')['patientid']
    .nunique()                           # Count unique patient IDs per demandpk
    .reset_index(name='n_patients')      # Rename the resulting column
)

# Step 2: Count how many demandpk fall into each category 
# (e.g., how many demandpk have 1 patient, how many have 2, etc.)
distribution = (
    demand_patient_counts['n_patients']
    .value_counts()                      # Count the frequency of each n_patients value
    .sort_index()                        # Sort by number of patients for readability
)

print(distribution)


## Resolution codes

In [ ]:
import re

# -------------------------------------------------------
# 1. Split combined codes (e.g. "34 | 28") into one code per row
# -------------------------------------------------------

# Split by '|' and clean whitespace
df_codes = (
    df_triaje['resolution_code_id']
    .astype(str)
    .str.split('|')
    .apply(lambda x: [item.strip() for item in x])  # remove spaces
    .explode()                                      # create one row per code
    .replace({'': None, 'nan': None, 'None': None}) # normalize missing values
)

# -------------------------------------------------------
# 2. Helper function to sort by numeric + alphabetical part
# -------------------------------------------------------
def split_code(code):
    """
    Splits a resolution code into numeric and alphabetical parts.
    Example: "34A" -> (34, "A"), "36" -> (36, "")
    """
    if code is None:
        return (float('inf'), "")  # send None to the bottom
    
    code = str(code).strip()
    
    # Capture digits + optional letters
    match = re.match(r"(\d+)([A-Za-z]*)", code)
    if match:
        num = int(match.group(1))
        let = match.group(2)
        return (num, let)
    else:
        # If unexpected format, send it to the bottom
        return (float('inf'), code)

# -------------------------------------------------------
# 3. Unique values sorted
# -------------------------------------------------------
unique_codes = df_codes.dropna().unique()
sorted_codes = sorted(unique_codes, key=split_code)

# -------------------------------------------------------
# 4. Value counts sorted according to numeric+letter order
# -------------------------------------------------------
vc = df_codes.value_counts(dropna=False)

# Create sorted index
sorted_index = sorted(vc.index, key=split_code)

# Reindex to apply sorted order
vc_sorted = vc.reindex(sorted_index)

# -------------------------------------------------------
# 5. Display results
# -------------------------------------------------------
vc_sorted


# DATA QUALITY

## Analysis of Missing

In [ ]:
kb.analyze_missing_values(df_triaje)